# Path Tracking for a Robot Manipulator
Objective: track a specified path with the end-effector of a robot manipulator.

# Modeling a Robot Manipulator
We use as state of the robot manipulator its joint positions and velocities:
$$ x = (q, \dot{q})$$

The control inputs are the joint accelerations $\ddot{q}$ so the system dynamics is a double integrator:
$$ \ddot{q} = u$$

The torque bounds are nonlinear constraints depending on both $x$ and $u$, because the joint torques need to be computed using the **inverse dynamics** function:
$$\tau_{min} \le \underbrace{M(q) u + h(x)}_{\text{inv_dyn}(q, \dot{q}, \ddot{q})} \le \tau_{max}$$

# Path Specification
We assume that the target path is specified as a function of the scalar variable $s\in[0,1]$. For example, if the path has the shape of the infinity symbol with center $c$ and ray $r$:
$$p_x(s) = c_x + r \, \cos(2\pi s)$$
$$p_y(s) = c_y + 0.5 \, r \, \sin(4\pi s)$$

In [1]:
import matplotlib.pyplot as plt
import numpy as np

def plot_infinity(t_init, t_final):
    r = 0.1
    t = np.linspace(t_init, t_final, 300)
    x = r * np.cos(2*np.pi*t)
    y = r * 0.5*np.sin(4*np.pi*t)
    plt.figure(figsize=(10,4))
    plt.plot(x, y, ' x')
    plt.xlim([-0.11, 0.11])
    plt.ylim([-0.06, 0.06])
    plt.show()

In [2]:
from ipywidgets import interact, fixed
interact(plot_infinity, t_init=fixed(0), t_final=(0,1, 0.02))

interactive(children=(FloatSlider(value=0.0, description='t_final', max=1.0, step=0.02), Output()), _dom_class…

<function __main__.plot_infinity(t_init, t_final)>

# Path Tracking vs Trajectory Tracking
When tracking a trajectory we know exactly where the end-effector should be at each time step, so we can simply penalize the distance to the reference trajectory.

When tracking a path instead we do not know where on the path the end-effector should be at a given time. What cost could we use then?

## Distance to the path
A straightforward choice could be to minimize the distance to the path, but:
* Computing the distance to an arbitrary path may be complex
* Remaining still on the path would be an optimal solution

## Introducing s as decision variable
We can introduce the trajectory $s(t)$ as an additional state variable, with single integrator dynamics controlled by the additional control input $w(t)$:
$$s_{k+1} = s_k + \Delta t \, w_k$$

Then we can ensure the robot follows the path by constraining the end-effector to be equal to the point on the path corresponding to the current value of $s$: 
$$ y(q) = p(s) $$
where:
$$p_x(s) = c_x + r \, \cos(2\pi s)$$
$$p_y(s) = c_y + 0.5 \, r \, \sin(4\pi s)$$
$$p_z(s) = c_z$$

Finally, to ensure progress along the path we can either maximize $s$ in the cost, or constrain $s$ to reach 1 by the end of the horizon.
$$s_N = 1$$

# Problem Formulation
This is the complete optimal control problem:
$$ \min_{X, U, S, W} \quad \sum_{k=0}^{N-1} (w_v ||\dot{q}_k||^2 + w_a ||u_k||^2 + w_w ||w_k||^2) + w_{final} ||x_N-x_{init}||^2
\qquad\qquad\qquad\qquad$$
$$\text{subject to} \quad 
q_{k+1} = q_k + \Delta t \, \dot{q}_k, \quad 
\dot{q}_{k+1} = \dot{q}_k + \Delta t \, u_k \quad 
\forall k \in [0, N-1] \qquad \qquad \qquad \quad$$
$$\qquad \qquad 
q^{min} \le q_{k} \le q^{max}, \quad 
-\dot{q}^{max} \le \dot{q}_{k} \le \dot{q}^{max} \quad 
\forall k \in [1, N]  \quad \qquad \qquad \qquad$$
$$\qquad \qquad 
\tau_{min} \le \text{inv_dyn}(q_k, \dot{q}_k, u_k) \le \tau_{max} \qquad\qquad
\forall k \in [0, N-1]  \qquad \qquad \qquad \quad$$
$$\qquad \qquad 
s_{k+1} = s_k + \Delta t \, w_k, \qquad \qquad \qquad \qquad
\forall k \in [0, N-1] \qquad \qquad \qquad \quad$$
$$ 0 \le w_k \le w_{max} \qquad\qquad\qquad\qquad \forall k \in [0, N-1]$$
$$ y(q_k)=p(s_k)\qquad\qquad\qquad \forall k \in [1, N]$$
$$ x_0 = x_{init} \qquad \qquad  \qquad \qquad \qquad \qquad$$
$$ s_{0} = 0, \qquad s_N = 1 \qquad\qquad\qquad\qquad \qquad \qquad \quad$$


where $y(q)$ is the position of the end-effector.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from adam.casadi.computations import KinDynComputations
import casadi as cs
from time import time as clock
from time import sleep

import orc.utils.plot_utils as plut
from example_robot_data.robots_loader import load
import orc.optimal_control.casadi_adam.conf_ur5 as conf_ur5

In [4]:
robot = load("ur5")
joints_name_list = [s for s in robot.model.names[1:]] # skip the first name because it is "universe"
nq = len(joints_name_list)  # number of joints
nx = 2*nq                   # size of the state variable

In [5]:
dt = 0.02           # OCP time step
N = 100              # number of OCP time steps
q0 = np.zeros(nq)   # initial joint configuration
dq0= np.zeros(nq)   # initial joint velocities
x_init = np.concatenate([q0, dq0])
w_max = 10
frame_name = "ee_link"
r_path = 0.2       # ray of the path

In [6]:
# create the dynamics function
q   = cs.SX.sym('q', nq)
dq  = cs.SX.sym('dq', nq)
ddq = cs.SX.sym('ddq', nq)
state = cs.vertcat(q, dq)
rhs    = cs.vertcat(dq, ddq)
f = cs.Function('f', [state, ddq], [rhs])

In [7]:
# create a Casadi inverse dynamics function
kinDyn = KinDynComputations(robot.urdf, joints_name_list)
H_b = cs.SX.eye(4)     # base configuration
v_b = cs.SX.zeros(6)   # base velocity
bias_forces = kinDyn.bias_force_fun()
mass_matrix = kinDyn.mass_matrix_fun()
# discard the first 6 elements because they are associated to the robot base
h = bias_forces(H_b, q, v_b, dq)[6:]
M = mass_matrix(H_b, q)[6:,6:]
tau = M @ ddq + h
inv_dyn = cs.Function('inv_dyn', [state, ddq], [tau])

In [8]:
# create a Casadi forward kinematics function
if(frame_name not in kinDyn.rbdalgos.model.links.keys()):
    print("ERROR. Frame name can only take values from this list")
fk_fun = kinDyn.forward_kinematics_fun(frame_name)
ee_pos = fk_fun(H_b, q)[:3, 3]
fk = cs.Function('fk', [q], [ee_pos])

y = fk(q0)
c_path = np.array([y[0]-r_path, y[1], y[2]]).squeeze() # center of the path

In [9]:
# pre-compute state and torque bounds
lbx = robot.model.lowerPositionLimit.tolist() + (-robot.model.velocityLimit).tolist()
ubx = robot.model.upperPositionLimit.tolist() + robot.model.velocityLimit.tolist()
tau_min = (-robot.model.effortLimit).tolist()
tau_max = robot.model.effortLimit.tolist()

In [10]:
# create all the decision variables
def create_decision_variables(N, nx, nu, lbx, ubx, w_max):
    opti = cs.Opti()
    X, U, S, W = [], [], [], []
    for k in range(N+1): 
        X += [opti.variable(nx)]
        opti.subject_to( opti.bounded(lbx, X[-1], ubx) )
        S += [opti.variable(1)]
        opti.subject_to( opti.bounded(0, S[-1], 1) )
    for k in range(N): 
        U += [opti.variable(nu)]
        W += [opti.variable(1)]
        opti.subject_to( opti.bounded(0, W[-1], w_max))
    return opti, X, U, S, W

In [11]:
def define_running_cost_and_dynamics(opti, X, U, S, W, N, dt, x_init, 
                                     c_path, r_path, w_p, w_v, w_a, w_w, tau_min, tau_max):
    opti.subject_to(X[0][:nq] == x_init[:nq])
    opti.subject_to(S[0] == 0.0)
    opti.subject_to(S[-1] == 1.0)
    cost = 0.0
    for k in range(N):     
        # add path constraints
        ee_pos = fk(X[k][:nq])
        opti.subject_to(ee_pos[0] == c_path[0] + r_path*cs.cos(2*np.pi*S[k]))
        opti.subject_to(ee_pos[1] == c_path[1] + r_path*0.5*cs.sin(4*np.pi*S[k]))
        opti.subject_to(ee_pos[2] == c_path[2])

        # Compute cost function        
        cost += w_v * X[k][nq:].T @ X[k][nq:] * dt
        cost += w_a * U[k].T @ U[k] * dt
        cost += w_w * W[k] * W[k] * dt

        # Add dynamics constraints
        opti.subject_to(X[k+1] == X[k] + dt * f(X[k], U[k]))
        opti.subject_to(S[k+1] == S[k] + dt * W[k])

        # Add torque constraints
        opti.subject_to( opti.bounded(tau_min, inv_dyn(X[k], U[k]), tau_max))
    return cost

In [12]:
# add the terminal cost and constraints
def define_terminal_cost_and_constraints(opti, X, S, c_path, r_path, w_final):
    ee_pos = fk(X[-1][:nq])
    opti.subject_to(ee_pos[0] == c_path[0] + r_path*cs.cos(2*np.pi*S[-1]))
    opti.subject_to(ee_pos[1] == c_path[1] + r_path*0.5*cs.sin(4*np.pi*S[-1]))
    opti.subject_to(ee_pos[2] == c_path[2])

    # make sure trajectory is cyclic
    cost = w_final * (X[-1]-X[0]).T @ (X[-1]-X[0])
#     opti.subject_to(X[-1] == x_init)
    return 0.0

In [13]:
def create_and_solve_ocp(N, nx, nq, lbx, ubx, dt, x_init, 
                        c_path, r_path, w_p, w_v, w_a, w_w, w_final, tau_min, tau_max, w_max):
    opti, X, U, S, W = create_decision_variables(N, nx, nq, lbx, ubx, w_max)
    running_cost = define_running_cost_and_dynamics(opti, X, U, S, W, N, dt, x_init, 
                                                    c_path, r_path, w_p, w_v, w_a, w_w, tau_min, tau_max)
    terminal_cost = define_terminal_cost_and_constraints(opti, X, S, c_path, r_path, w_final)
    opti.minimize(running_cost + terminal_cost)
    
    # Create the optimization problem
    opts = {"ipopt.print_level": 0, "print_time": 0, "ipopt.tol": 1e-4,
            "ipopt.constr_viol_tol": 1e-4, "ipopt.compl_inf_tol": 1e-4}
    opti.solver("ipopt", opts)
    
    t0 = time.time()
    sol = opti.solve()
    t1 = time.time()
    print("Solver time", t1-t0)
    
    return sol, X, U, S, W

In [14]:
def extract_solution(sol, X, U, S, W):
    x_sol = np.array([sol.value(X[k]) for k in range(N+1)]).T
    ddq_sol = np.array([sol.value(U[k]) for k in range(N)]).T
    s_sol = np.array([sol.value(S[k]) for k in range(N+1)]).T
    w_sol = np.array([sol.value(W[k]) for k in range(N)]).T
    q_sol = x_sol[:nq,:]
    dq_sol = x_sol[nq:,:]
    # compute joint torques trajectory
    tau = np.zeros((nq, N))
    for i in range(N):
        tau[:,i] = inv_dyn(x_sol[:,i], ddq_sol[:,i]).toarray().squeeze()
    # compute end-effector trajectory
    ee = np.zeros((3, N+1))
    for i in range(N+1):
        ee[:,i] = fk(x_sol[:nq,i]).toarray().squeeze()
    # compute desired end-effector trajectory
    ee_des = np.zeros((3, N+1))
    for i in range(N+1):
        ee_des[:,i] = np.array([c_path[0] + r_path*cs.cos(2*np.pi*s_sol[i]),
                                c_path[1] + r_path*0.5*cs.sin(4*np.pi*s_sol[i]),
                                c_path[2]])
    return q_sol, dq_sol, ddq_sol, tau, ee, ee_des, s_sol, w_sol

In [15]:
from orc.utils.robot_wrapper import RobotWrapper
from orc.utils.robot_simulator import RobotSimulator
from orc.utils.viz_utils import addViewerSphere, applyViewerConfiguration
import time

r = RobotWrapper(robot.model, robot.collision_model, robot.visual_model)
simu = RobotSimulator(conf_ur5, r)
simu.init(q0, dq0)
simu.display(q0)

REF_SPHERE_RADIUS = 0.02
EE_REF_SPHERE_COLOR = np.array([1, 0, 0, .5])

def display_motion(q_traj, ee_des_traj):
    for i in range(N+1):
        t0 = time.time()
        simu.display(q_traj[:,i])
        addViewerSphere(r.viz, 'world/ee_ref_'+str(i), REF_SPHERE_RADIUS, EE_REF_SPHERE_COLOR)
        applyViewerConfiguration(r.viz, 'world/ee_ref_'+str(i), ee_des_traj[:,i].tolist()+[0,0,0,1.])
        t1 = time.time()
        if(t1-t0 < dt):
            sleep(dt - (t1-t0))

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7002/static/


In [16]:
def solve_and_display(log_w_p, log_w_v, log_w_a, log_w_w, log_w_final):
    sol, X, U, S, W = create_and_solve_ocp(N, nx, nq, lbx, ubx, dt, x_init, c_path, r_path, 
                                     10**log_w_p, 10**log_w_v, 10**log_w_a, 10**log_w_w, 10**log_w_final, 
                                     tau_min, tau_max, w_max)
    q_sol, dq_sol, u_sol, tau, ee, ee_des, s_sol, w_sol = extract_solution(sol, X, U, S, W)
    for i in range(3):
        display_motion(q_sol, ee_des)

In [17]:
r.viz.viewer.jupyter_cell()

In [18]:
from ipywidgets import interact, fixed
interact(solve_and_display, log_w_p=fixed(-6), log_w_v=(-6,-1,1), log_w_a=(-6,-1,1), log_w_w=(-6,0,1), log_w_final=fixed(0))

interactive(children=(IntSlider(value=-4, description='log_w_v', max=-1, min=-6), IntSlider(value=-4, descript…

<function __main__.solve_and_display(log_w_p, log_w_v, log_w_a, log_w_w, log_w_final)>

In [19]:
def solve_and_plot(log_w_p, log_w_v, log_w_a, log_w_w, log_w_final):
    sol, X, U, S, W = create_and_solve_ocp(N, nx, nq, lbx, ubx, dt, x_init, c_path, r_path, 
                                     10**log_w_p, 10**log_w_v, 10**log_w_a, 10**log_w_w, 10**log_w_final, 
                                     tau_min, tau_max, w_max)
    q_sol, dq_sol, u_sol, tau, ee, ee_des, s_sol, w_sol = extract_solution(sol, X, U, S, W)
    tt = np.linspace(0, (N+1)*dt, N+1)

    plt.figure(figsize=(10, 4))
    plt.plot([tt[0], tt[-1]], [0, 1], ':', label='straight line', alpha=0.7)
    plt.plot(tt, s_sol, label='s', alpha=0.7)
    plt.xlabel('Time [s]')
    plt.legend()
    plt.grid(True)
    
    plt.figure(figsize=(10, 4))
    plt.plot(ee_des[0,:].T, ee_des[1,:].T, 'r x', label='EE des', alpha=0.7)
    plt.plot(ee[0,:].T, ee[1,:].T, 'k o', label='EE', alpha=0.7)
    plt.xlabel('End-effector pos x [m]')
    plt.ylabel('End-effector pos y [m]')
    plt.legend()
    plt.grid(True)
    
    plt.figure(figsize=(10, 4))
    for i in range(3):
        plt.plot(tt, ee_des[i,:].T, ':', label=f'EE des {i}', alpha=0.7)
        plt.plot(tt, ee[i,:].T, label=f'EE {i}', alpha=0.7)
    plt.xlabel('Time [s]')
    plt.ylabel('End-effector pos [m]')
    plt.legend()
    plt.grid(True)

    plt.figure(figsize=(10, 4))
    for i in range(dq_sol.shape[0]):
        plt.plot(tt, dq_sol[i,:].T, label=f'dq {i}', alpha=0.7)
    plt.xlabel('Time [s]')
    plt.ylabel('Joint velocity [rad/s]')
    plt.legend()
    plt.grid(True)

    plt.figure(figsize=(10, 4))
    for i in range(tau.shape[0]):
        plt.plot(tt[:-1], tau[i,:].T, label=f'tau {i}', alpha=0.7)
    plt.xlabel('Time [s]')
    plt.ylabel('Joint torque [Nm]')
    plt.legend()
    plt.grid(True)
    plt.show()

In [20]:
interact(solve_and_plot, log_w_p=fixed(-6), log_w_v=(-6,-1,1), log_w_a=(-6,-1,1), log_w_w=(-6,0,1), log_w_final=fixed(0))

interactive(children=(IntSlider(value=-4, description='log_w_v', max=-1, min=-6), IntSlider(value=-4, descript…

<function __main__.solve_and_plot(log_w_p, log_w_v, log_w_a, log_w_w, log_w_final)>